# 🦅 Áquila Pro — Master Multimodal Neural Hub & Swarm AI Trainer
## Fine-Tuning de Agentes Escolares, Geração Multimodal de Imagens, PhET Colorado & Método Kumon

Este notebook executa o motor neural de alta performance em **GPU NVIDIA (A100 80GB ou T4 16GB)** no Google Colab.

### 🎯 O que este ambiente realiza:
1. **Fine-Tuning QLoRA (4-bit)** de 6 agentes especializados (Professor Storyteller, Marketing/Leads, Autoridade Pedagógica, Tutor Socrático, Avaliação Formativa e Mediador Escolar).
2. **Geração Multimodal de Imagens (SDXL-Turbo / Diffusers)** em estilo Anime/Super-Heróis para suporte à abstração espacial de estudantes em questões de física e ciências.
3. **Engenharia Reversa PhET Colorado**: Catálogo de 165 simulações interativas HTML5 e 85 simulações Java legadas.
4. **Motor de Micro-Drills Kumon (L1, L2, L3)**: Elimina sobrecarga da memória de trabalho (Sweller) e destrava a matemática básica.
5. **Servidor FastAPI + Túnel Cloudflare**: Conecta o Colab diretamente ao seu Áquila Desktop (Windows) e Mobile.

In [ ]:
# 1. Verificação de Hardware NVIDIA & Suporte a Tensor Cores
!nvidia-smi

import torch
print(f"\n🚀 PyTorch: {torch.__version__}")
print(f"⚡ CUDA Disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🔥 Dispositivo Ativo: {gpu_name} ({vram:.2f} GB VRAM)")
    print(f"✨ Suporte a Bfloat16: {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ ATENÇÃO: Nenhuma GPU detectada. Vá em 'Ambiente de execução' > 'Alterar tipo' e ative GPU (T4 ou A100).")

In [ ]:
# 2. Instalação Automatizada da Stack Neural Completa
!pip install -q --upgrade pip
!pip install -q transformers peft accelerate bitsandbytes datasets trl
!pip install -q diffusers invisible-watermark
!pip install -q fastapi uvicorn pydantic httpx nest_asyncio pycloudflared

# Baixa o binário oficial do Cloudflare Tunnel para Linux x64
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ Stack neural instalada e Cloudflare Tunnel configurado!")

In [ ]:
# 3. Clonagem / Atualização do Repositório Áquila Pro & Carga dos Datasets
!git clone https://github.com/andrejrunb/-quila-pro-v-1.0.0.git aquila_repo || (cd aquila_repo && git pull origin main)

import json, os
from pathlib import Path

data_path = Path("aquila_repo/colab/data_finetuning")
print(f"📁 Datasets disponíveis em {data_path}:")
for f in data_path.glob("*.jsonl"):
    with open(f, 'r', encoding='utf-8') as jf:
        lines = len(jf.readlines())
    print(f"  • {f.name} ({lines} amostras de treino)")

phet_cat = Path("aquila_repo/colab/phet_physics_simulations_catalog.json")
if phet_cat.exists():
    with open(phet_cat, 'r', encoding='utf-8') as pf:
        sims = json.load(pf)
    print(f"\n🎮 Catálogo PhET Colorado carregado: {len(sims)} simulações de Física em HTML5!")

In [ ]:
# 4. Motor Multimodal de Imagens Didáticas (SDXL-Turbo / Diffusers)
# Gera diagramas visuais em estilo anime/super-herói para que o aluno consiga abstrair o problema de física
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image

print("🎨 Carregando modelo SDXL-Turbo na GPU...")
image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    variant="fp16" if torch.cuda.is_available() else None
)
if torch.cuda.is_available():
    image_pipe.to("cuda")
print("✅ SDXL-Turbo pronto para gerar imagens em 1 a 2 passos de inferência!")

def gerar_diagrama_fisica(prompt_conceito, output_name="diagrama_fisica.png"):
    """Gera uma imagem de alta resolução a partir do prompt pedagógico."""
    prompt_completo = f"clean anime educational illustration, {prompt_conceito}, scientific labels and vectors in neon colors, 4k high resolution, studio ghibli aesthetic"
    print(f"🖼️ Gerando: '{prompt_completo[:80]}...' [1-2 passos]")
    image = image_pipe(prompt=prompt_completo, num_inference_steps=2, guidance_scale=0.0).images[0]
    image.save(output_name)
    print(f"💾 Salvo com sucesso em: {output_name}")
    return image

# Teste rápido de geração visual
sample_img = gerar_diagrama_fisica("young astronaut superhero floating between Earth and Moon with glowing gravitational force vectors labeled Fg")
sample_img.resize((400, 400))

In [ ]:
# 5. Motor de Engenharia Reversa REA de Java PhET (.jar -> .java -> TypeScript)
# Utiliza o descompilador CFR 0.152 para dissecar simuladores antigos e transpor para TypeScript
import subprocess, zipfile
from pathlib import Path

cfr_jar = Path("aquila_repo/colab/cfr.jar")
print(f"☕ Descompilador CFR 0.152 localizado: {cfr_jar.exists()}")

def dissecar_simulacao_java(jar_path, output_dir="src_decompilado"):
    """Descompila um arquivo JAR do PhET para código Java legível."""
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    print(f"⚙️ Descompilando {jar_path} com CFR...")
    subprocess.run(["java", "-jar", str(cfr_jar), jar_path, "--outputdir", str(out), "--silent", "true"], check=True)
    java_files = list(out.rglob("*.java"))
    print(f"✅ Descompilação finalizada: {len(java_files)} classes Java reconstruídas com sucesso!")
    return java_files

print("💡 Execute dissecar_simulacao_java('caminho/para/simulacao.jar') para extrair equações de qualquer simulação legada!")

In [ ]:
# 6. Treinamento Fine-Tuning QLoRA Multi-LoRA dos Agentes Especializados
# Permite treinar qualquer um dos agentes escolares com aceleração por GPU
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct" # Ou 'meta-llama/Meta-Llama-3.1-8B-Instruct'
print(f"🧠 Preparando carregamento quantizado em 4-bit de {BASE_MODEL}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("✅ Configurações de QLoRA 4-bit preparadas!")
print("💡 Para treinar um agente específico, execute o script: !python aquila_repo/colab/aquila_multiagent_finetuner.py")

In [ ]:
# 7. Servidor FastAPI & Túnel Cloudflare para Conexão ao Áquila Desktop & Mobile
import subprocess, threading, time

print("🚀 Iniciando servidor FastAPI Multi-Agente em background...")
server_proc = subprocess.Popen([
    "python", "aquila_repo/colab/aquila_multimodal_supertrainer.py"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(3)

print("🌐 Gerando URL Pública Segura via Cloudflare Tunnel...")
cf_proc = subprocess.Popen([
    "cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in cf_proc.stdout:
    if "trycloudflare.com" in line:
        for word in line.split():
            if "trycloudflare.com" in word:
                public_url = word.strip()
                print("\n" + "=" * 75)
                print(f"🔗 TÚNEL PÚBLICO CLOUDFLARE ATIVO: {public_url}")
                print("👉 Copie e cole esta URL no Áquila Pro Desktop (Electron) ou Mobile!")
                print("=" * 75 + "\n")
                break
        break